# CachePilot Phase 1 Colab 验收

本 notebook 可在 Colab 单 GPU 上执行真实 TorchExecutor、渐进压测、故障注入和回归检查。所有结果写入 Google Drive，便于下载回仓库。

注意：`progressive_load.py` 只产生 progressive artifacts；正式策略矩阵还需要每次 run 的 `manifest.json`、`trace.jsonl`、`requests.jsonl` 和 `summary.json`。不能把 progressive 文件重命名为正式协议文件。

## 1. 挂载仓库


In [ ]:
from google.colab import drive
import os, pathlib
drive.mount('/content/drive', force_remount=True)
WORKDIR = pathlib.Path('/content/drive/MyDrive/CachePilot')
if not WORKDIR.is_dir():
    raise FileNotFoundError(WORKDIR)
os.chdir(WORKDIR)
print(WORKDIR)

## 2. 安装并固定依赖

Colab runtime 版本必须与仓库基线一致；安装失败时停止，不降级为 fake/CPU 结果。

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements/requirements-gpu.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest==8.3.5', 'httpx==0.28.1'], check=True)
os.environ['PYTHONPATH'] = str(WORKDIR)

## 3. GPU 环境门禁


In [ ]:
import json, torch, transformers
from benchmarks.gpu_environment import collect_environment, validate_environment
environment = collect_environment(WORKDIR)
failures = validate_environment(environment, require_torch=True)
print(json.dumps(environment, ensure_ascii=False, indent=2))
if failures: raise RuntimeError('; '.join(failures))
assert torch.cuda.is_available()
print('GPU_PYTHON_PREFLIGHT=PASS')

## 4. API/SSE/取消/超时回归


In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/integration/test_gateway_api.py', 'tests/integration/test_vllm_gateway.py', 'tests/unit/test_torch_executor.py'], cwd=WORKDIR, text=True)
if result.returncode: raise RuntimeError('API/executor regression failed')
print('API_REGRESSION=PASS')

## 5. 启动真实 Torch 服务

四种策略应分别使用同一份 workload；每次只保留一个服务进程。`team-a` 是默认授权 tenant。

In [ ]:
import subprocess, time, urllib.request
strategy = {'admission': 'strict', 'scheduler': 'fcfs', 'prefix_mode': 'blind'}
server_log = WORKDIR / 'runs' / 'colab-server.log'
log_handle = server_log.open('w')
server = subprocess.Popen([sys.executable, 'main.py', '--host', '127.0.0.1', '--port', '8000', '--admission', strategy['admission'], '--scheduler', strategy['scheduler'], '--prefix-mode', strategy['prefix_mode']], cwd=WORKDIR, stdout=log_handle, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if json.load(urllib.request.urlopen('http://127.0.0.1:8000/readyz'))['status'] == 'ready': break
    except Exception: time.sleep(1)
else: raise RuntimeError('GPU service did not become ready')
print('GPU_SERVICE=READY')

## 6. 渐进压测（四种策略分别执行）

先运行最小并发修复回归，再运行完整矩阵。每个点至少三次，失败点不能被解释为安全点。

In [ ]:
import subprocess, sys
progressive_dir = WORKDIR / 'runs' / 'colab-progressive-strict-fcfs-blind'
cmd = [sys.executable, 'benchmarks/progressive_load.py', '--base-url', 'http://127.0.0.1:8000', '--tenant-id', 'team-a', '--output-dir', str(progressive_dir), '--contexts', '32,128,512,2048,4096,8192', '--concurrencies', '1,2,4,8', '--max-tokens', '16', '--repetitions', '3', '--warmup']
subprocess.run(cmd, cwd=WORKDIR, check=True)
print(progressive_dir / 'progressive_report.json')

对另外三种策略，停止服务后修改第 5 个单元的 `strategy`，重新启动，再把输出目录改成：

- `runs/colab-progressive-adaptive-fcfs-blind`
- `runs/colab-progressive-strict-wfq-blind`
- `runs/colab-progressive-strict-wfq-aware`

当前 TorchExecutor 是 single-request batch，不执行 static-vs-continuous。

## 7. 故障注入


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'benchmarks.chaos_validation', '--output', 'runs/phase1-chaos/chaos_report.json'], cwd=WORKDIR, check=True)
print((WORKDIR / 'runs/phase1-chaos/chaos_report.json').read_text())

## 8. 正式 strategy_matrix run 的条件

本 notebook 的 progressive 输出不能直接用于 `strategy_matrix.py`。要得到 `MATRIX_VALID`，必须由正式实验记录器为每个策略保存：

```text
manifest.json
trace.jsonl
requests.jsonl
summary.json
```

每个策略需要 1 个 warm-up + 3 个 measured run。若 Colab 中没有生成这些正式协议文件，本节只应报告 `MATRIX_NOT_READY`，不能复制或改名 progressive 文件冒充。

In [ ]:
from pathlib import Path
formal_root = WORKDIR / 'runs' / 'required-matrix'
formal_runs = sorted(p for p in formal_root.glob('*') if p.is_dir()) if formal_root.is_dir() else []
required = ('manifest.json', 'trace.jsonl', 'requests.jsonl', 'summary.json')
missing = [str(p) for p in formal_runs if any(not (p / f).is_file() for f in required)]
if len(formal_runs) < 16 or missing:
    print('MATRIX_NOT_READY: need 16 complete formal run directories')
else:
    print('MATRIX_FORMAL_ARTIFACTS=READY')

## 9. 最终门禁


In [ ]:
import subprocess, sys
formal_root = WORKDIR / 'runs' / 'required-matrix'
if formal_root.is_dir() and len(list(formal_root.glob('*'))) >= 16:
    for run in sorted(p for p in formal_root.glob('*') if p.is_dir()):
        subprocess.run([sys.executable, 'benchmarks/analyze.py', '--manifest', str(run/'manifest.json'), '--trace', str(run/'trace.jsonl'), '--requests', str(run/'requests.jsonl'), '--output', str(run/'summary.json')], cwd=WORKDIR, check=True)
    subprocess.run([sys.executable, '-m', 'benchmarks.strategy_matrix', '--metric', 'ttft_ms', '--output', str(formal_root/'matrix.json'), *map(str, sorted(p for p in formal_root.glob('*') if p.is_dir()))], cwd=WORKDIR, check=True)
else:
    print('MATRIX_NOT_READY: progressive artifacts are not formal strategy runs')